In [5]:
%pip -q install duckdb pyarrow

from google.colab import drive
from pathlib import Path
import os

import duckdb
import pandas as pd
import pyarrow.parquet as pq

drive.mount("/content/drive", force_remount=False)

DATA_DIR = Path("/content/drive/MyDrive/Language Detection")
PARQUET_PATH = DATA_DIR / "sessions_lang_transcript_2026-08-23_2026-08-24.parquet"
TEMP_DIR = Path("/content/duckdb_tmp")

if not PARQUET_PATH.is_file():
    raise FileNotFoundError(PARQUET_PATH)

TEMP_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"SET threads = {max(1, min(os.cpu_count() or 4, 8))}")
con.execute("SET memory_limit = '2GB'")
con.execute(f"SET temp_directory = '{TEMP_DIR.as_posix()}'")
con.execute("SET preserve_insertion_order = false")

parquet = pq.ParquetFile(PARQUET_PATH)
metadata = parquet.metadata

print("File :", PARQUET_PATH)
print("Size :", f"{PARQUET_PATH.stat().st_size / 1024**2:.2f} MB")
print("Rows :", f"{metadata.num_rows:,}")
print("Cols :", metadata.num_columns)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File : /content/drive/MyDrive/Language Detection/sessions_lang_transcript_2026-08-23_2026-08-24.parquet
Size : 469.49 MB
Rows : 3,469
Cols : 12


In [6]:
LANGUAGES = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt": "Portuguese",
    "es": "Spanish",
    "ru": "Russian",
}

MIN_WORDS = 4
MIN_SEGMENTS = 90
SESSIONS_PER_LANGUAGE = 5

language_sql = ", ".join(f"'{code}'" for code in LANGUAGES)

con.execute(
    f"""
    CREATE OR REPLACE TABLE session_stats AS
    SELECT
        gamesession_id,
        user_id,
        game_name,
        url,
        model_type,
        TRY_CAST(created_at AS TIMESTAMP) AS created_at,
        lang_detected AS language_code,
        lang_probability,
        list_count(
            list_filter(
                transcript_segments,
                segment ->
                    segment.words IS NOT NULL
                    AND len(segment.words) >= {MIN_WORDS}
            )
        ) AS segment_count
    FROM read_parquet('{PARQUET_PATH.as_posix()}')
    WHERE
        lang_detected IN ({language_sql})
        AND transcript_segments IS NOT NULL
        AND len(transcript_segments) >= {MIN_SEGMENTS}
    """
)

con.execute(
    f"""
    CREATE OR REPLACE TABLE selected_sessions AS
    WITH ranked AS (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY language_code
                ORDER BY
                    segment_count ASC,
                    created_at DESC NULLS LAST,
                    gamesession_id DESC
            ) AS session_rank
        FROM session_stats
        WHERE segment_count >= {MIN_SEGMENTS}
    )
    SELECT
        language_code,
        session_rank,
        gamesession_id,
        user_id,
        game_name,
        url,
        model_type,
        created_at,
        lang_probability,
        segment_count
    FROM ranked
    WHERE session_rank <= {SESSIONS_PER_LANGUAGE}
    """
)

selected_sessions = con.execute(
    """
    SELECT *
    FROM selected_sessions
    ORDER BY
        language_code,
        session_rank
    """
).df()

selected_sessions.insert(
    0,
    "language",
    selected_sessions["language_code"].map(LANGUAGES),
)

display(selected_sessions)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,language,language_code,session_rank,gamesession_id,user_id,game_name,url,model_type,created_at,lang_probability,segment_count
0,German,de,1,141268064,467269,Naraka,https://www.twitch.tv/videos/2854317619,gen10,2026-08-23 23:29:07,0.8291,98
1,German,de,2,141264210,399263,Call of Duty: Modern Warfare 4,https://www.twitch.tv/videos/2854157061,warfare4,2026-08-23 18:39:26,0.9072,107
2,German,de,3,141265651,687197,Escape from Tarkov,https://www.twitch.tv/videos/2854208944,gen10,2026-08-23 21:53:10,0.9199,114
3,German,de,4,141236998,833542,COD: Warzone3-2,https://www.twitch.tv/videos/2853489925,warfare2,2026-08-23 03:22:23,0.6826,115
4,German,de,5,141238626,812794,COD: Warzone3-2,https://www.twitch.tv/videos/2853620823,warfare2,2026-08-23 04:09:16,0.9678,118
5,English,en,1,141266766,395792,Phasmophobia,https://www.twitch.tv/videos/2854265768,gen10,2026-08-23 22:19:05,0.9463,90
6,English,en,2,141245486,773042,COD: Modern Warfare III,https://www.youtube.com/watch?v=UAUU-ciTr08,warfare3,2026-08-23 12:30:09,0.9277,90
7,English,en,3,141250470,615244,Helldivers 2,https://www.twitch.tv/videos/2853894951,gen4,2026-08-23 09:36:17,0.9976,90
8,English,en,4,141210406,466455,COD: Warzone3-2,https://www.twitch.tv/videos/2852833829,warfare2,2026-08-23 00:10:17,0.8442,90
9,English,en,5,141236416,100090,Call of Duty: Modern Warfare 4,https://www.twitch.tv/videos/2853520407,warfare4,2026-08-23 09:08:50,0.6890,91


In [7]:
con.execute(
    f'''
    CREATE OR REPLACE TABLE selected_segments AS
    WITH source AS (
        SELECT
            p.gamesession_id,
            p.lang_detected AS language_code,
            p.transcript_segments
        FROM read_parquet('{PARQUET_PATH.as_posix()}') AS p
        INNER JOIN selected_sessions AS s
            ON p.gamesession_id = s.gamesession_id
            AND p.lang_detected = s.language_code
    ),
    exploded AS (
        SELECT
            gamesession_id,
            language_code,
            generate_subscripts(transcript_segments, 1) AS segment_index,
            UNNEST(transcript_segments) AS segment
        FROM source
    )
    SELECT
        gamesession_id,
        language_code,
        segment_index,
        TRIM(segment.text) AS segment_text,
        len(segment.words) AS word_count
    FROM exploded
    WHERE
        segment.words IS NOT NULL
        AND len(segment.words) >= {MIN_WORDS}
        AND segment.text IS NOT NULL
        AND TRIM(segment.text) <> ''
    '''
)

selected_segment_counts = con.execute(
    '''
    SELECT
        language_code,
        gamesession_id,
        COUNT(*) AS selected_segment_count,
        MIN(word_count) AS min_word_count
    FROM selected_segments
    GROUP BY
        language_code,
        gamesession_id
    ORDER BY
        language_code,
        gamesession_id
    '''
).df()

display(selected_segment_counts)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,language_code,gamesession_id,selected_segment_count,min_word_count
0,de,141236998,115,4
1,de,141238626,118,4
2,de,141264210,107,4
3,de,141265651,114,4
4,de,141268064,98,4
5,en,141210406,90,4
6,en,141236416,91,4
7,en,141245486,90,4
8,en,141250470,90,4
9,en,141266766,90,4


In [8]:
session_validation = con.execute(
    f"""
    WITH language_checks AS (
        SELECT
            language_code,
            COUNT(*) AS session_count,
            MIN(segment_count) AS min_segment_count,
            COUNT(*) = {SESSIONS_PER_LANGUAGE} AS has_five_sessions,
            MIN(segment_count) >= {MIN_SEGMENTS} AS all_sessions_have_min_segments
        FROM selected_sessions
        GROUP BY language_code
    ),
    word_checks AS (
        SELECT
            language_code,
            MIN(word_count) AS min_word_count,
            MIN(word_count) >= {MIN_WORDS} AS all_segments_have_min_words
        FROM selected_segments
        GROUP BY language_code
    ),
    order_checks AS (
        SELECT
            language_code,
            bool_and(
                next_segment_count IS NULL
                OR segment_count <= next_segment_count
            ) AS smallest_segment_count_first
        FROM (
            SELECT
                language_code,
                session_rank,
                segment_count,
                lead(segment_count) OVER (
                    PARTITION BY language_code
                    ORDER BY session_rank
                ) AS next_segment_count
            FROM selected_sessions
        )
        GROUP BY language_code
    )
    SELECT
        l.language_code,
        l.session_count,
        l.min_segment_count,
        w.min_word_count,
        l.has_five_sessions,
        l.all_sessions_have_min_segments,
        w.all_segments_have_min_words,
        o.smallest_segment_count_first,
        (
            l.has_five_sessions
            AND l.all_sessions_have_min_segments
            AND w.all_segments_have_min_words
            AND o.smallest_segment_count_first
        ) AS all_checks_passed
    FROM language_checks AS l
    INNER JOIN word_checks AS w USING (language_code)
    INNER JOIN order_checks AS o USING (language_code)
    ORDER BY language_code
    """
).df()

session_validation.insert(
    0,
    "language",
    session_validation["language_code"].map(LANGUAGES),
)

display(session_validation)

if len(session_validation) != len(LANGUAGES):
    raise ValueError("Validation did not cover all target languages")

if not session_validation["all_checks_passed"].all():
    raise ValueError("Validation failed")

print("All validation checks passed.")

,language,language_code,session_count,min_segment_count,min_word_count,has_five_sessions,all_sessions_have_min_segments,all_segments_have_min_words,smallest_segment_count_first,all_checks_passed
0,German,de,5,98,4,True,True,True,True,True
1,English,en,5,90,4,True,True,True,True,True
2,Spanish,es,5,94,4,True,True,True,True,True
3,French,fr,5,130,4,True,True,True,True,True
4,Portuguese,pt,5,91,4,True,True,True,True,True
5,Russian,ru,5,90,4,True,True,True,True,True


All validation checks passed.
